# Run SAGA DNA Design

This notebook runs the DNA design experiments with configurable levels of automation (co-pilot, semi-pilot, auto-pilot), along with comprehensive configurations, best model selection, and proper logging.

## Device Requirements

- Must be able to run docker, because the coding agent will need it to debug and test scorers that it creates.
- GPUs are optional. If your scorers rely on running some deep learning models, having GPUs would be much quicker.
- Better be a Linux machine. Not sure if you'd meet any issue on other OSs.

## Part 1: Set up Configurations for an Experiment

### Step 1.1: Import Everything 

In [ ]:
import os
import sys
import json
from pprint import pprint
from datetime import datetime
from pathlib import Path

In [ ]:
# This changes the working directory to the root directory of the project.
current_dir = os.getcwd()
split_path = current_dir.split("/")
project_root_index = split_path.index("SAGA")
project_root_dir = "/".join(split_path[:project_root_index + 1])
os.chdir(project_root_dir)
if project_root_dir not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Current working directory:", os.getcwd())

In [ ]:
# Import the core modules
from scileo_agent.core.config import create_config
from scileo_agent.core.orchestrator import OptimizationOrchestrator
from scileo_agent.core.registry import list_registered_modules, get_scorer, list_scorers, register_mcp_module, get_serializer, list_serializers, reset_scorer_manager, get_scorer_metadata, ScorerManager
from scileo_agent.core.data_models import Candidate, Population, Objective
from scileo_agent.utils.logging import get_logger, setup_logging

In [ ]:
# Import the logging utils
logger = setup_logging(level="DEBUG")  # Setup the logger

In [ ]:
# Import the shared modules (e.g., the planner, the scorer creator, the analyzer)
# 🔴 Please make sure that your serializer is imported here.
from modules.shared import planner, scorer_creator, analyzer, knowledge_manager, serializer

In [ ]:
# Check the available scorers
# At this point, there should not be any scorers registered
scorer_names = list_scorers()
print(scorer_names)
if len(scorer_names) > 0:
    raise RuntimeError("There should not be any scorers registered at this point. Check the above imports and make sure no scorer is imported along with them.")

In [ ]:
# Get the serializer
# 🔴 Needs your customization

serializer_name = "dna_serializer"

serializer = get_serializer(serializer_name)
if serializer is None:
    raise RuntimeError(f"Serializer '{serializer_name}' not found. Registered serializers: {list_serializers()}. If your serializer is not listed, make sure it's imported in `modules/shared/serializer/__init__.py`.")

In [ ]:
# Import your optimizer
# 🔴 Needs your customization

from modules.dna_design.optimizer import *

In [ ]:
# Check the available modules
# 🔴 Needs your check: Make sure all modules to use are imported, especially your custom optimizer

pprint(list_registered_modules())

In [ ]:
# Import the MCP scorers
# 🔴 Needs your customization

# Put ONLY the initial objectives' MCP scorer module paths for your task here
# They will be registered in the scorer library
mcp_scorer_paths = [
    'modules/dna_design/scorer_mcp/dna_enhancer_scorers_mcp',
]

# You don't have to change anything below

module_names = set()
for scorer_path in mcp_scorer_paths:
    module_name = Path(scorer_path).name
    module_names.add(module_name)

if len(module_names) != len(mcp_scorer_paths):
    raise RuntimeError("Duplicate module names found in mcp_scorer_paths. Please make sure each module has a unique name.")

for scorer_path in mcp_scorer_paths:
    register_mcp_module(scorer_path, serializer_name=serializer_name)

### Step 1.2: Set up Configurations

🔴 Please carefully check the description of each argument and set its value. The existing values are recommended but feel free to change them as required.

In [ ]:
# Run ID
exp_level = 3  # Experiment level: 1 (Co-pilot), 2 (Semi-pilot), or 3 (Autopilot)

# 🔴 Configs that you must change for your task
run_name = "dna_enhancer_design_001" + f"_level{exp_level}"  # Could be your task name

# Other configs
run_id = f"{run_name}-{datetime.now().strftime('%Y%m%d%H%M%S')}"

print(f"run_id: {run_id}")

In [ ]:
# Set up the level-specific configurations

if exp_level == 1:
    analyzer_enable_human_feedback = True
    planner_enable_human_feedback = True
elif exp_level == 2:
    analyzer_enable_human_feedback = True
    planner_enable_human_feedback = False
elif exp_level == 3:
    analyzer_enable_human_feedback = False
    planner_enable_human_feedback = False
else:
    raise ValueError("exp_level must be 1, 2, or 3.")

In [ ]:
# Loop configs

# 🔴 Configs that you may want to change for your task
max_iterations = 2 # The maximum number of iterations to run. The "2" here is just for testing. Make sure it's large enough, say 20. Analyzer may stop the loop earlier if convergence is detected.
random_candidate_ratio = 0.0  # The ratio of candidates to be replaced before being fed into the optimizer every iteration (except iteration 1)

# Other configs
max_objective_planning_retries = 3  # The maximum number of retries for objective planning, if any planned objective cannot be implemented
return_all_candidates = True  # Whether to return and evaluate all candidates across all iterations in result.all_candidates_population

In [ ]:
# Planner configs

# 🔴 Configs that you may want to change for your task
planner_model_name = "openai/gpt-5-2025-08-07"  # The LLM for the planner. Don't change the model, but you can change the provider
requires_objective_weights = False  # Whether the planner needs to provide weights for each objective
support_filter = False  # Whether the planner should propose filter objectives
support_population_wise = False  # Whether the planner should propose population-wise objectives
max_objectives = 3  # The maximum number of objectives to propose for each iteration. None means no limit. Set it if your optimizer cannot handle too many objectives
do_high_level_planning = False  # Whether to do high-level planning before proposing objectives in the first iteration

# Other configs
planner_max_llm_retries = 3  # The maximum number of retries for LLM calls if one fails to get a valid objective plan
use_context_information = "first_iteration"  # The value can be "first_iteration", "all_iterations", "disabled". This controls whether to provide context information to the planner, and in which iterations. You can also set it to "all_iterations" if you want to provide the context information over and over again to the planner in all iterations


In [ ]:
# Scorer creator configs

# 🔴 Configs that you may want to change for your task
scorer_creator_model_name = "openai/gpt-5-2025-08-07"  # The LLM for the scorer creator for matching existing scorers, not for the coding agent. Don't change the model, but you can change the provider
enable_llm_scorer_creation = True  # Whether to enable the coding agent to create new scorers when no existing scorers can match the proposed objectives
coding_agent_model_name = 'anthropic/claude-sonnet-4-5-20250929'  # Don't change the model, but you can change the provider to "anthropic" (Claude API), "bedrock" (AWS) or "claude_code" (your local Claude Code client, which can use your Claude Pro/Max account). Recommending not using "claude_code" to avoid unexpected limit exceeded issues.
reference_module_paths = mcp_scorer_paths  # The reference module paths to provide to the coding agent as examples. We provide all your registered MCP scorer modules here by default
use_potential_matched_scorers_as_references = True  # Whether to automatically use the potential matched existing scorers (judged by an LLM) as references even if they are not in the reference_module_paths (but included in the registered scorers mcp_scorer_paths)

# Other configs
coding_workspace_path = f"runs/{run_id}/coding_workspace"  # The workspace path for the coding agent
generated_scorer_library_path = f"runs/{run_id}/generated_scorers"  # The path to save the generated scorer modules
scorer_library_subfolder = None  # The subfolder under the generated_scorer_library_path to save the generated scorer modules. If None, the modules will be saved directly under the generated_scorer_library_path
scorer_creator_dev = False  # Whether to enable the developer mode for the scorer creator. If enabled, it would be more strict on exceptions and may raise exceptions instead of recovering from them
coding_agent_run_in_docker = True  # Whether to run the coding agent in a Docker container.
scorer_creator_max_llm_retries = 3  # The maximum number of retries for LLM calls if one fails to get a valid response
coding_agent_max_parallel_scorer_creation = 3  # The maximum number of scorers that the coding agent can implement in parallel.
enable_name_matching = True  # Whether to enable the name-based matching
enable_llm_matching = True  # Whether to enable the LLM-based matching by scorer descriptions

In [ ]:
# print available optimizers

pprint(list_registered_modules(module_type="optimizer")['optimizer'])

In [ ]:
# Optimizer configs

# 🔴 Configs that you must set for your task
optimizer_module_name = 'llm_dna_enhancer_optimizer'  # The name of the optimizer module to use. It has to be one of the registered optimizers (check the above printed list)
optimizer_module_version = "1.0.0"  # The version of the optimizer module to use. It has to be one of the registered versions of the selected optimizer (check the above printed list)
optimizer_config = {  # Put your optimizer-specific configs here
    "n_rounds": 5,
    "batch_size": 20,
    "seq_length": 200,
    "num_initial_seqs": 5000,
    "use_diversity_for_filtering": True,
    "diversity_filtering_threshold": 0.5,
}
optimizer_model_name = "openai/gpt-4.1-2025-04-14"  # The LLM for the optimizer. Set None if the optimizer doesn't use any LLM.

In [ ]:
# Analyzer configs

# 🔴 Configs that you may want to change for your task
analyzer_model_name = "openai/gpt-5-2025-08-07"
refusal_detection_model_name = "openai/gpt-4.1-nano-2025-04-14"  # The LLM for refusal detection. Don't change the model, but you can change the provider
candidate_analyzer_model_name = "anthropic/claude-sonnet-4-5-20250929"  # Don't change the model, but you can change the provider: claude_code, anthropic, bedrock
candidate_analyzer_run_in_docker = True  # Recommend to set True, because the docker environment installs many dependencies for analysis
candidate_analyzer_enable_domain_tools = True  # Enable domain-specific tools for candidate analysis
candidate_analyzer_tool_selection_model = 'anthropic/claude-sonnet-4-5-20250929'  # Model for tool selection. Don't change the model, but you can change the provider.

# Other configs
population_save_dir = f"runs/{run_id}/populations_for_analysis"  # The directory to save the population data for each iteration
analyzer_max_llm_retries = 3  # The maximum number of retries for LLM calls if one fails to get a valid analysis
enable_candidate_analysis = True  # Whether to enable candidate-level analysis
candidate_analyzer_workspace = f"runs/{run_id}/candidate_analyzer_workspace"  # The workspace path for the candidate analyzer
enable_refusal_detection = True  # Whether to enable refusal detection in the analyzer
candidate_analyzer_tooluniverse_path = '/opt/tooluniverse-env' if candidate_analyzer_run_in_docker else './tooluniverse-env' # Don't need to change

In [ ]:
# Create the configurations
# You don't need to change anything here. It just assembles all the configs above into a single config object.
config = create_config(
    run_id=run_id,
    run_name=run_name,
    loop_config={
        "max_iterations": max_iterations,
        "max_objective_planning_retries": max_objective_planning_retries,
        "random_candidate_ratio": random_candidate_ratio,
        "return_all_candidates": return_all_candidates,
    },
    module_configs={
        "planner": {
            "config": {
                'requires_objective_weights': requires_objective_weights,
                'support_filter': support_filter,
                'support_population_wise': support_population_wise,
                'max_objectives': max_objectives,
                'do_high_level_planning': do_high_level_planning,
                'max_llm_retries': planner_max_llm_retries,
                "use_context_information": use_context_information,
                "enable_human_feedback": planner_enable_human_feedback,
            },
            "llm_config": {"model_name": planner_model_name}
        },
        "scorer_creator": {
            "config": {
                "enable_llm_scorer_creation": enable_llm_scorer_creation,
                "coding_agent_model_name": coding_agent_model_name,
                "reference_module_paths": reference_module_paths,
                "use_potential_matched_scorers_as_references": use_potential_matched_scorers_as_references,
                "coding_workspace_path": coding_workspace_path,
                "generated_scorer_library_path": generated_scorer_library_path,
                "scorer_library_subfolder": scorer_library_subfolder,
                "dev": scorer_creator_dev,
                "coding_agent_run_in_docker": coding_agent_run_in_docker,
                "max_llm_retries": scorer_creator_max_llm_retries,
                "coding_agent_max_parallel_scorer_creation": coding_agent_max_parallel_scorer_creation,
                "enable_name_matching": enable_name_matching,
                "enable_llm_matching": enable_llm_matching,
            },
            "llm_config": {"model_name": scorer_creator_model_name},
        },
        "optimizer": {
            "module_name": optimizer_module_name,
            "module_version": optimizer_module_version,
            "config": optimizer_config,
            "llm_config": {"model_name": optimizer_model_name} if optimizer_model_name is not None else None,
        },
        "analyzer": {
            "config": {
                "analyzer_model_name": analyzer_model_name,
                "refusal_detection_model_name": refusal_detection_model_name,
                "candidate_analyzer_workspace": candidate_analyzer_workspace,
                "candidate_analyzer_model_name": candidate_analyzer_model_name,
                "candidate_analyzer_run_in_docker": candidate_analyzer_run_in_docker,
                "candidate_analyzer_enable_domain_tools": candidate_analyzer_enable_domain_tools,
                "candidate_analyzer_tool_selection_model": candidate_analyzer_tool_selection_model,
                "population_save_dir": population_save_dir,
                "analyzer_max_llm_retries": analyzer_max_llm_retries,
                "enable_candidate_analysis": enable_candidate_analysis,
                "enable_refusal_detection": enable_refusal_detection,
                "candidate_analyzer_tooluniverse_path": candidate_analyzer_tooluniverse_path,
            },
            "llm_config": {"model_name": analyzer_model_name}
        }
    }
)

In [ ]:
# Print all the configurations and let's check if everything is correct
print(config)

In [ ]:
# Create the orchestrator
orchestrator = OptimizationOrchestrator(
    config,
    run_name=run_name,
    run_id=run_id,
)

## Part 2: Run SAGA

### Step 2.1: Prepare Your Inputs

In [ ]:
# 🔴 Prepare your inputs

# Clearly describe the optimization goal
optimization_goal = "Generate a set of cell-type-specific enhancers for the HepG2 cell line, each with a length of 200 base pairs."

# Provide additional context information to help the planner better understand your task and propose better objectives
# Such as the task background, the constraints, and your initial ideas about the objectives
# No need to mention that you'll provide an initial set of objectives, because the planner will always get them from the user in the first iteration
# You don't have to necessarily follow the below example. You can provide any information that you think is helpful for the planner to understand your task and propose better objectives
context_information = "For this task, the enhancers should be specific to the HepG2 cell line, meaning they should drive high expression in HepG2 cells while minimizing expression in other cell lines (e.g., K562 and SKNSH). The sequences should also be diverse to cover a broad range of potential enhancer activities. You can consider including objectives related to known enhancer motifs and stability of the DNA sequences. The optimizer will automatically enforce the length constraint, so do not propose any objectives related to enhancer length."

In [ ]:
# Print out the registered objectives

print("Registered scorers:")
for objective_name in list_scorers():
    metadata = get_scorer_metadata(objective_name)
    print(f"- {objective_name}: {metadata['description']}")

In [ ]:
# Specify initial objectives

# 🔴 Put the names of the initial objectives and their optimization directions here
# They have to be among the registered scorers printed above
# For filter objectives (if any), the optimization direction should always be None
initial_objective_names_and_optimization_directions = {
    "dna_hepg2_enhancer_MPRA_expression": "maximize",
    "dna_k562_enhancer_MPRA_expression": "minimize",
    "dna_sknsh_enhancer_MPRA_expression": "minimize",
}

# You don't need to change anything below. It checks the validity of the initial objectives and creates Objective instances for them.

# Make sure that the corresponding MCP modules do not contain other objectives than the initial ones
scorer_manager = ScorerManager()
module_set = set()
for objective_name in initial_objective_names_and_optimization_directions:
    module_name = scorer_manager.mcp_scorer_to_module[objective_name]
    module_set.add(module_name)
module_scorers_set = set()
for module_name in module_set:
    module_scorers = scorer_manager.mcp_module_to_scorers[module_name]
    for scorer in module_scorers:
        module_scorers_set.add(scorer)
if len(initial_objective_names_and_optimization_directions) != len(module_scorers_set):
    raise RuntimeError("The initial objectives correspond to MCP modules that contain other objectives than the initial ones. Please make sure each MCP module only contains the initial objective(s) by splitting an MCP module into separate ones. Otherwise, the coding agent would be able to refer to those other objectives when generating new scorers, which leads to information leakage.")

# Check the initial objectives' types
type_set = set()
for objective_name in initial_objective_names_and_optimization_directions:
    metadata = get_scorer_metadata(objective_name)
    type_set.add(metadata['type'])

if not support_filter and "filter" in type_set:
    raise RuntimeError("One or more initial objectives are of type 'filter', but the planner's `support_filter` is set to `False`.")
if not support_population_wise and "population_wise" in type_set:
    raise RuntimeError("One or more initial objectives are of type 'population_wise', but the planner's `support_population_wise` is set to `False`.")

# Create Objective instances for the initial objectives

initial_objectives = []
for initial_objective_name, optimization_direction in initial_objective_names_and_optimization_directions.items():
    scorer = get_scorer(initial_objective_name)
    if scorer is None:
        raise RuntimeError(f"Initial objective scorer '{initial_objective_name}' not found. Registered scorers: {list_scorers()}.")
    metadata = get_scorer_metadata(initial_objective_name)
    objective = Objective(
        name=initial_objective_name,
        description=metadata['description'],
        type=metadata['type'],
        scorer=scorer,
        optimization_direction=optimization_direction,
    )
    initial_objectives.append(objective)


In [ ]:
# 🔴 Create initial population in some way
# Change it to your own way

optimizer = orchestrator.optimizer
# For testing purposes, we limit the initial population size (500) to speed up the iterations
# In real runs, you may want to use all the candidates in the initial population
initial_population = Population(candidates=await optimizer.create_random_candidates(500))

In [ ]:
# Boom! Run!
try:
    result = await orchestrator.run(
        optimization_goal, 
        context_information, 
        serializer_name=serializer_name,
        initial_objectives=initial_objectives, 
        initial_population=initial_population, 
    )
finally:
    # Make sure MCP scorers' docker containers are properly cleaned up
    reset_scorer_manager()

In [ ]:
raise RuntimeError(
    "This cell is a guard to prevent subsequent cells from running automatically. "
    "Once the above cell completes, please manually run the cells below to process the results."
)

In [ ]:
# Print per-iteration statistics from the optimization result
print("=" * 70)
print("OPTIMIZATION RESULTS — Per-Iteration Summary")
print("=" * 70)
print(f"  Run ID   : {result.run_id}")
print(f"  Status   : {result.status}")
print(f"  Iterations: {result.total_generations}")
if result.termination_reason:
    print(f"  Termination: {result.termination_reason}")
print()

for i in range(0, result.total_generations + 1):
    population = orchestrator.knowledge_manager.get_population(i)
    if population is None or population.is_empty:
        continue

    label = "Iteration 0 (Initial Population)" if i == 0 else f"Iteration {i}"
    print(f"{'─' * 70}")
    print(f"  {label}")
    print(f"{'─' * 70}")
    print(f"  Candidates : {population.size}")

    # Collect all objective/score names present in this population's candidates
    objective_names = sorted({
        key
        for candidate in population.candidates
        for key in candidate.scores.keys()
    })

    if objective_names:
        print(f"  Scores:")
        col_w = max(len(n) for n in objective_names)
        for obj_name in objective_names:
            mean, std, none_count = population.get_regular_score_mean_and_std(obj_name)
            mean_str = f"{mean:>10.4f}" if mean is not None else "       N/A"
            std_str  = f"{std:>9.4f}"   if std  is not None else "      N/A"
            valid    = population.size - none_count
            print(f"    {obj_name:<{col_w}}  mean={mean_str}  std={std_str}  "
                  f"valid={valid}/{population.size}  missing={none_count}")
    else:
        print("  (no scores available)")
    print()

print("=" * 70)
if result.final_population:
    print(f"  Final population size : {result.final_population.size}")
if result.all_candidates_population:
    print(f"  All candidates (all iterations) : {result.all_candidates_population.size}")
print("=" * 70)
